# 1. Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings

# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# 2. Defining the Product Information and Location

In [2]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][9]
print(search_text)
Source="Summit"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'
filename=Source+'ProductLinks_'+search_text+'.xlsx'
df1=pd.read_excel(IFolder+'\\'+filename)
df1

                                  Product Name
0                                Ignition Coil
1                      Windshield Washer Pumps
2                        coupler trailer locks
3          Adjustable Trailer Hitch Ball Mount
4                               Vacuum Cleaner
5                                   Spark Plug
6              bluetooth Enabled Trailer locks
7                          Power Steering Hose
8   Power Steering Pressure Line Hose Assembly
9                       Washer Fluid Reservoir
10          Hitch Ball Mount with Weight Scale
11                               LED Headlamps
12                             LED flashlights
13                   Fiberglass Tonneau Covers
14                    Aluminium Tonneau Covers
15                     Hardfold Tonneau Covers
16                            Spark Plug Wires
17                 washer fluid reservoir tank
18                   Non Automotive Gas Struts
Washer Fluid Reservoir


,Links,Name,Sl.No
0,https://www.summitracing.com/parts/rnb-603-131,Dorman Windshield Washer Fluid Reservoirs 603-131,1
1,https://www.summitracing.com/parts/rnb-603-072,Dorman Windshield Washer Fluid Reservoirs 603-072,2
2,https://www.summitracing.com/parts/rnb-603-162,Dorman Windshield Washer Fluid Reservoirs 603-162,3
3,https://www.summitracing.com/parts/rnb-603-018,Dorman Windshield Washer Fluid Reservoirs 603-018,4
4,https://www.summitracing.com/parts/gmk-4012-24...,Goodmark Windshield Washer Fluid Reservoirs GM...,5
...,...,...,...
288,https://www.summitracing.com/parts/bdy-au1288104,Body Parts Windshield Washer Reservoirs AU1288104,296
289,https://www.summitracing.com/parts/bdy-lx1288117,Body Parts Windshield Washer Reservoirs LX1288117,297
290,https://www.summitracing.com/parts/ado-23362222,ACDelco GM Genuine Parts Windshield Washer Res...,298
291,https://www.summitracing.com/parts/ado-84763238,ACDelco GM Genuine Parts Windshield Washer Res...,299


In [3]:
df1.loc[1,"Name"]

'Dorman Windshield Washer Fluid Reservoirs 603-072'

In [4]:
links=[]
for i in range(len(df1)):
    if " " in df1.loc[i,"Name"]:
        links.append(df1['Links'][i])

In [6]:
length=len(links)
length

293

# 3. Setting Webdriver and Website Specific Information

In [7]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.get(links[1])
driver.maximize_window()

wait=WebDriverWait(driver, 5)

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [8]:
cols =['Sl.No','Attributes'] #,'Review_Mentions'
df = pd.DataFrame(columns=cols)
count=0

In [9]:
from IPython.display import clear_output 
import numpy as np
from datetime import timedelta
import datetime 
timestamp=[]
timediff=[]
def timeremaining(balanceitem,i):
    timestamp.insert(i,datetime.datetime.now())
    if i>=1:
        diff=timestamp[i]-timestamp[i-1]
        timediff.insert(i,diff.total_seconds())		
        remainingtime=np.median(timediff)*balanceitem
        millis=int(remainingtime)		
        #print(millis)
        seconds=(millis)%60
        seconds = int(seconds)
        minutes=(millis/(60))%60
        minutes = int((minutes)) #math.floor
        hours=(millis/(60*60))%24
        hours=int(hours)
        print(f'Item looped: {i+1} \nItems Remining: {balanceitem-1} and \nTime Remaining: {hours}:{minutes}:{seconds}')
        etc=datetime.datetime.now()+timedelta(hours=hours,minutes=minutes,seconds=seconds)
        print(f'Estimated time of completion:{etc}')        
        clear_output(wait=True)

In [20]:
length=240

In [12]:
for i in range(250,length):
    driver.get(links[i])
    df.loc[count,'Sl.No']=i+1
    sleep(1)
    df.loc[count,'Name']=driver.find_element(By.CSS_SELECTOR, '[class="part-detail-title"]').text
    try:
        df.loc[count,'Current Price']=driver.find_element(By.CSS_SELECTOR, '[class="price"]').text
    except:
        pass
    df.loc[count,'Product']=search_text
    df.loc[count,'Part Number']=driver.find_element(By.CSS_SELECTOR, '[class="part-detail-title-number"]').text.split(': ')[1].split("-")[1]
    Alist=[]    
    try:
        Aelements=wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME,"large-6")))
        Velements=driver.find_elements(By.CLASS_NAME,'large-18')        
        for a, v in zip(Aelements, Velements):
            Alist.append(a.text+v.text)
        df.at[count,'Attributes']=Alist
    except:
        pass
    try:
        element=driver.find_element(By.CSS_SELECTOR,'[class="value-title"]')
        df.loc[count,'Rating']=float(element.get_attribute('title').split(' out')[0])
    except:
        df.loc[count,'Rating']=0
    df.loc[count,'Details']=driver.find_element(By.CSS_SELECTOR,'[class="part-detail-description"]').text
    df.loc[count,'No of Ratings']=driver.find_element(By.CSS_SELECTOR, '[class="review-count"]').text.replace('( ','').replace(' )','')
    df.loc[count,'Links']=links[i]
    df.loc[count,'Source']=Source
    count=count+1
    balanceitem=length-i
    timeremaining(balanceitem,i)

Item looped: 293 
Items Remining: 0 and 
Time Remaining: 0:0:3
Estimated time of completion:2024-02-09 14:03:14.320066


In [13]:
print(df.shape)
df.head()

(293, 11)


,Sl.No,Attributes,Name,Current Price,Product,Part Number,Rating,Details,No of Ratings,Links,Source
0,1,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-131,$48.95,Washer Fluid Reservoir,603,3.0,Is your windshield washer fluid leaking out of...,3,https://www.summitracing.com/parts/rnb-603-131,Summit
1,2,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-072,$50.99,Washer Fluid Reservoir,603,3.0,Is your windshield washer fluid leaking out of...,1,https://www.summitracing.com/parts/rnb-603-072,Summit
2,3,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-162,$35.95,Washer Fluid Reservoir,603,0.0,Is your windshield washer fluid leaking out of...,Write the First Review,https://www.summitracing.com/parts/rnb-603-162,Summit
3,4,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-018,$51.95,Washer Fluid Reservoir,603,4.0,Is your windshield washer fluid leaking out of...,6,https://www.summitracing.com/parts/rnb-603-018,Summit
4,5,"[Brand:Goodmark, Manufacturer's Part Number:GM...",Goodmark Windshield Washer Fluid Reservoirs GM...,$17.99,Washer Fluid Reservoir,4012,5.0,Goodmark windshield washer fluid reservoirs ar...,1,https://www.summitracing.com/parts/gmk-4012-24...,Summit


# 5. Post Processing Data and Exporting

In [14]:
df['Brand']=df['Attributes'].apply(lambda x: str(x).replace('[','').replace(']','').replace('\'', '').replace(' ','')).str.split('Brand:',expand=True)[1].str.split(',',expand=True)[0]

In [15]:
df['UPC Code']=df['Attributes'].apply(lambda x: str(x).replace('[','').replace(']','').replace('\'', '').replace(' ','')).str.split('UPC:',expand=True)[1].str.split(',',expand=True)[0]

In [16]:
df['No of Ratings']=df['No of Ratings'].str.replace('Write the First Review','0').astype(int)
df['Sl.No']=df['Sl.No'].astype(int)
df['Current Price']=df['Current Price'].str.replace('$','').astype(float)
df['Product']=search_text
df

,Sl.No,Attributes,Name,Current Price,Product,Part Number,Rating,Details,No of Ratings,Links,Source,Brand,UPC Code
0,1,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-131,48.95,Washer Fluid Reservoir,603,3.0,Is your windshield washer fluid leaking out of...,3,https://www.summitracing.com/parts/rnb-603-131,Summit,Dorman,019495239376
1,2,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-072,50.99,Washer Fluid Reservoir,603,3.0,Is your windshield washer fluid leaking out of...,1,https://www.summitracing.com/parts/rnb-603-072,Summit,Dorman,019495439585
2,3,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-162,35.95,Washer Fluid Reservoir,603,0.0,Is your windshield washer fluid leaking out of...,0,https://www.summitracing.com/parts/rnb-603-162,Summit,Dorman,019495453567
3,4,"[Brand:Dorman, Manufacturer's Part Number:603-...",Dorman Windshield Washer Fluid Reservoirs 603-018,51.95,Washer Fluid Reservoir,603,4.0,Is your windshield washer fluid leaking out of...,6,https://www.summitracing.com/parts/rnb-603-018,Summit,Dorman,019495439400
4,5,"[Brand:Goodmark, Manufacturer's Part Number:GM...",Goodmark Windshield Washer Fluid Reservoirs GM...,17.99,Washer Fluid Reservoir,4012,5.0,Goodmark windshield washer fluid reservoirs ar...,1,https://www.summitracing.com/parts/gmk-4012-24...,Summit,Goodmark,00615343781211
...,...,...,...,...,...,...,...,...,...,...,...,...,...
288,289,[Brand:Coast to Coast International Body Parts...,Body Parts Windshield Washer Reservoirs AU1288104,104.99,Washer Fluid Reservoir,AU1288104,0.0,Body Parts windshield washer reservoirs are hi...,0,https://www.summitracing.com/parts/bdy-au1288104,Summit,CoasttoCoastInternationalBodyParts,None
289,290,[Brand:Coast to Coast International Body Parts...,Body Parts Windshield Washer Reservoirs LX1288117,92.99,Washer Fluid Reservoir,LX1288117,0.0,Body Parts windshield washer reservoirs are hi...,0,https://www.summitracing.com/parts/bdy-lx1288117,Summit,CoasttoCoastInternationalBodyParts,191275069592
290,291,"[Brand:ACDelco, Manufacturer's Part Number:233...",ACDelco GM Genuine Parts Windshield Washer Res...,38.99,Washer Fluid Reservoir,23362222,0.0,ACDelco GM Genuine Parts windshield washer res...,0,https://www.summitracing.com/parts/ado-23362222,Summit,ACDelco,00195491143234
291,292,"[Brand:ACDelco, Manufacturer's Part Number:847...",ACDelco GM Genuine Parts Windshield Washer Res...,59.99,Washer Fluid Reservoir,84763238,0.0,ACDelco GM Genuine Parts windshield washer res...,0,https://www.summitracing.com/parts/ado-84763238,Summit,ACDelco,00193468175820


In [17]:
cols=["Sl.No",
"Name",
"Product",
"Current Price",
"Rating",
"No of Ratings",
"Details",
"Attributes",
"Brand",
"Part Number",
"UPC Code",
"Links",
"Source"
]
df=df[cols]

In [18]:
df=df.drop_duplicates(subset=['Links'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 293 entries, 0 to 292
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sl.No          293 non-null    int32  
 1   Name           293 non-null    object 
 2   Product        293 non-null    object 
 3   Current Price  293 non-null    float64
 4   Rating         293 non-null    float64
 5   No of Ratings  293 non-null    int32  
 6   Details        293 non-null    object 
 7   Attributes     293 non-null    object 
 8   Brand          293 non-null    object 
 9   Part Number    293 non-null    object 
 10  UPC Code       263 non-null    object 
 11  Links          293 non-null    object 
 12  Source         293 non-null    object 
dtypes: float64(2), int32(2), object(9)
memory usage: 29.8+ KB


In [19]:
dfAtt=df[['UPC Code','Attributes']]
dfAtt=dfAtt.explode('Attributes')
dfAtt[['Attributes', 'Value']] = dfAtt['Attributes'].str.split(':',1, expand=True)

In [20]:
summary_df = pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count').reset_index()
summary_df =summary_df.sort_values(by='Value',ascending=False)
summary_df=summary_df.rename(columns ={'Value':'No Products contains this Attribute'})
summary_df

,Attributes,No Products contains this Attribute
0,Brand,293
1,Manufacturer's Part Number,293
3,Part Type,293
4,Product Line,293
5,Quantity,293
9,Summit Racing Part Number,293
10,UPC,263
8,Reservoir Material,149
7,Reservoir Color,133
2,Notes,36


In [21]:
with pd.ExcelWriter(OFolder+'\\'+f'{Source}ProductDetails_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Raw')
    summary_df.to_excel(writer,index=False, sheet_name='Attribute_Summary')

# 99. Archived Codes